# dim_commenter_scd1 (SCD 1)

One row per comment author. Natural key `commenter_nk = COALESCE(author_channel_id, 'name:' || author_display_name)`; `commenter_key = xxhash64(commenter_nk)` (same as `dim_commenter`).

Task `dim_commenter_scd1` of the job "2. gold (poc tables)"; widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.dim_commenter_scd1"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
stg_comments = spark.table(f"{silver}.stg_comments")
stg_comments_latest = spark.table(f"{silver}.stg_comments_latest")

commenter_nk = F.coalesce(F.col("author_channel_id"), F.concat(F.lit("name:"), F.col("author_display_name")))
comments = stg_comments_latest.withColumn("commenter_nk", commenter_nk).filter(F.col("commenter_nk").isNotNull())

## Incremental: authors with a comment read in the window

Their first / last comment is then recomputed over **all** their comments.

In [ ]:
if load_type == "incremental":
    changed = (stg_comments
               .filter((F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date))
               .select(commenter_nk.alias("commenter_nk")).distinct())
    comments = comments.join(changed, "commenter_nk", "left_semi")

# Transformation

One row per author: latest name (newest comment) + first / last comment time.

In [ ]:
from pyspark.sql.window import Window

per_author = Window.partitionBy("commenter_nk")
newest_first = per_author.orderBy(F.col("published_at").desc(), F.col("comment_id"))

result_df = (
    comments
    .withColumn("rn", F.row_number().over(newest_first))
    .withColumn("first_comment_at", F.min("published_at").over(per_author))
    .withColumn("last_comment_at", F.max("published_at").over(per_author))
    .filter("rn = 1")
    .select(
        F.xxhash64("commenter_nk").alias("commenter_key"),
        F.col("commenter_nk").cast("string").alias("commenter_nk"),
        F.col("author_channel_id").cast("string").alias("author_channel_id"),
        F.col("author_display_name").cast("string").alias("author_display_name"),
        F.col("first_comment_at").cast("timestamp").alias("first_comment_at"),
        F.col("last_comment_at").cast("timestamp").alias("last_comment_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["commenter_nk"]
result_df = add_checksums(result_df, primary_key, exclude=["commenter_key"])

## Create table

In [ ]:
path = f"{gold}.dim_commenter_scd1"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        commenter_key BIGINT, commenter_nk STRING, author_channel_id STRING, author_display_name STRING,
        first_comment_at TIMESTAMP, last_comment_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Comment author dimension, SCD 1 (poc checksum method). commenter_key = xxhash64(commenter_nk), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("commenter_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (commenter_key, author_display_name, etl_date, etl_by) VALUES (-1, 'Unknown', current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT commenter_nk) FROM {path} WHERE commenter_nk IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate commenter_nk"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")